# Gradient Boosting — Amphitheatre Classification & Outside Detection

**Owner:** Person 5 (Modeling & Evaluation)  
**Project:** AmphiLocator — ENSIA ML 2025–2026

This notebook trains and evaluates a **gradient boosting** classifier for indoor amphitheatre localization using **GPS-derived features only** (no label leakage).

## Two main tasks

1. **Multi-class amphitheatre classification** — predict which of **Amphi 1–8** a student is in (`label_enc` 0–7).
2. **Outside detection** — identify when the student is **outside all amphitheatres** (`label_enc` 8), using two post-hoc strategies compared against the raw multi-class model.

## Pipeline overview

Load cleaned splits → select leakage-free features → add GPS interactions → baseline → tune → evaluate → cross-validate → interpret → outside detection → save artifacts.

In [ ]:
# Cell 2 — Imports
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Load data

We use the **feature-engineered, leakage-safe** splits produced by notebooks 02–03 (`*_ready.csv`). Each row is one GPS collection with distance-to-centroid and quality features already computed; centroids were fit on **train only** upstream.

In [ ]:
# Cell 3 — Load data
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = (NOTEBOOK_DIR / "..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train" / "train_ready.csv")
val = pd.read_csv(DATA_DIR / "val" / "val_ready.csv")
test = pd.read_csv(DATA_DIR / "test" / "test_ready.csv")

print("Paths:")
print(f"  train: {DATA_DIR / 'train' / 'train_ready.csv'}")
print(f"  val:   {DATA_DIR / 'val' / 'val_ready.csv'}")
print(f"  test:  {DATA_DIR / 'test' / 'test_ready.csv'}")
print()

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name}: shape {df.shape}")
    print(df["label_enc"].value_counts().sort_index())
    print()

## Feature selection (clean, no leakage)

We use only features that can be computed at **inference time from GPS**, without access to the collection UI or the true label.

| Included | Rationale |
|----------|------------|
| Distance to each amphi centroid, `dist_nearest`, `dist_2nd`, `dist_gap` | Core spatial signal |
| GPS quality (`accuracy_mean`, `log_accuracy`, bins, flags) | Noise handling |
| `sample_count` | Reading density |

| **Dropped** | **Why** |
|-------------|--------|
| `is_outside` | Used when building `target_label` → **label leakage** |
| Seat features (`seat_*`, `seat_zone_id`, `has_seat`) | **Gameable** by students; not reliable at deploy time |
| `nearest_amphi_enc` | Argmin-distance amphi name encoded → **direct label proxy** |
| `*_scaled` columns | **Redundant** for tree models (monotonic splits on raw distances) |
| Raw lat/lon | Replaced by distance features upstream |
| `hour_sin`, `hour_cos` | Collection-schedule signal, not location (see note below) |

Time features (`hour_sin`, `hour_cos`) were removed because they capture the data collection schedule of 3rd-year students, not the physical location. In production, any amphitheatre can be occupied at any hour — using time would cause the model to learn a timetable rather than spatial boundaries.

In [ ]:
# Cell 4 — Feature selection
FEATURES = [
    "dist_Amphi_1", "dist_Amphi_2", "dist_Amphi_3", "dist_Amphi_4",
    "dist_Amphi_5", "dist_Amphi_6", "dist_Amphi_7", "dist_Amphi_8",
    "dist_nearest", "dist_2nd", "dist_gap",
    "accuracy_mean", "log_accuracy", "accuracy_bin", "high_accuracy_flag",
    "sample_count",
]
TARGET = "label_enc"
OUTSIDE_LABEL = 8  # alphabetical LabelEncoder: Outside is class index 8

print(f"Target column: {TARGET}")
print(f"Base feature count: {len(FEATURES)}")
print("Features:")
for f in FEATURES:
    print(f"  - {f}")

missing = {name: [c for c in FEATURES if c not in df.columns] for name, df in
             [("train", train), ("val", val), ("test", test)]}
any_missing = any(missing.values())
if any_missing:
    for split, cols in missing.items():
        if cols:
            print(f"\nWARNING — missing in {split}: {cols}")
    raise ValueError("Some FEATURES are missing from the loaded CSVs.")
else:
    print("\nAll base features present in train, val, and test.")

## Interaction features (GPS-only)

These combinations capture **signal quality × spatial ambiguity** — e.g. uncertain GPS when the point lies between two rooms. They use only GPS-derived columns; **no label-derived information**.

In [ ]:
# Cell 5 — Interaction features
INTERACTION_FEATURES = [
    "dist_nearest_x_logacc",
    "dist_gap_x_highacc",
    "dist_nearest_sqrt",
]


def add_interactions(df: pd.DataFrame) -> pd.DataFrame:
    """Add legitimate GPS interaction columns (in place on a copy)."""
    out = df.copy()
    out["dist_nearest_x_logacc"] = out["dist_nearest"] * out["log_accuracy"]
    out["dist_gap_x_highacc"] = out["dist_gap"] * out["high_accuracy_flag"]
    out["dist_nearest_sqrt"] = np.sqrt(out["dist_nearest"].clip(lower=0))
    return out


train = add_interactions(train)
val = add_interactions(val)
test = add_interactions(test)

FEATURES = FEATURES + INTERACTION_FEATURES
print(f"Total features after interactions: {len(FEATURES)}")
print(FEATURES)

## Baseline model

A **default** `HistGradientBoostingClassifier` (no hyperparameter search) establishes a reference point on the **validation** set before tuning. We expect tuning to improve macro-F1, especially on minority classes.

In [ ]:
# Cell 6 — Baseline model
X_train = train[FEATURES]
y_train = train[TARGET]
X_val = val[FEATURES]
y_val = val[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

baseline = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

y_val_pred = baseline.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average="macro")

print("Baseline (default hyperparameters) — validation set")
print(f"  Accuracy:  {val_acc:.4f}")
print(f"  Macro F1:  {val_f1:.4f}")
print()
print(classification_report(y_val, y_val_pred, zero_division=0))

## Hyperparameter tuning

`GridSearchCV` runs on the **training set only** with 5-fold stratified CV and `f1_macro` scoring (balanced across the nine classes). The validation and test sets stay untouched until final evaluation.

In [ ]:
# Cell 7 — Hyperparameter tuning
param_grid = {
    "learning_rate": [0.05, 0.1, 0.2],
    "max_iter": [100, 200, 300],
    "max_depth": [3, 5, 7],
}

cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
base_estimator = HistGradientBoostingClassifier(random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    base_estimator,
    param_grid,
    cv=cv_tune,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
    refit=True,
)

print("Starting grid search on training set only …")
grid_search.fit(X_train, y_train)

print("\nBest parameters:", grid_search.best_params_)
print(f"Best CV f1_macro: {grid_search.best_score_:.4f}")

best_params = grid_search.best_params_

## Tuned model evaluation

We refit the best estimator on **train** and report metrics on **val** and **test** separately. Per-class metrics highlight weak classes (e.g. Amphi 7).

In [ ]:
# Cell 8 — Tuned model evaluation
best_model = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE,
    **best_params,
)
best_model.fit(X_train, y_train)


def evaluate_split(name: str, X, y, model) -> dict:
    """Accuracy, macro F1, and full classification report for one split."""
    y_pred = model.predict(X)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print(f"\n{'=' * 60}")
    print(f"{name} — Accuracy: {acc:.4f}  |  Macro F1: {f1:.4f}")
    print("=" * 60)
    print(classification_report(y, y_pred, zero_division=0))
    return {"split": name, "accuracy": acc, "macro_f1": f1, "y_pred": y_pred}


val_results = evaluate_split("Validation", X_val, y_val, best_model)
test_results = evaluate_split("Test", X_test, y_test, best_model)

# Confusion matrix — test set
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_results["y_pred"], ax=ax, cmap="Blues", values_format="d"
)
ax.set_title("Tuned GBM — Test set confusion matrix")
plt.tight_layout()
cm_path = FIGURES_DIR / "gbm_confusion_matrix_test.png"
plt.savefig(cm_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved: {cm_path}")

# Per-class F1 on test set (from classification report), sorted low → high
report_dict = classification_report(
    y_test, test_results["y_pred"], output_dict=True, zero_division=0
)
f1_by_class = pd.Series(
    {k: report_dict[k]["f1-score"] for k in report_dict if k.isdigit()}
).sort_values()
print("\nPer-class F1 (test set), low → high:")
for label, score in f1_by_class.items():
    print(f"  Class {label}: {score:.4f}")

### Validation vs test

**Val and test scores should be similar** — both are held-out stratified splits from the same collection protocol. A large gap (e.g. val ≈ 0.95, test ≈ 0.70) suggests **overfitting** to train or split-specific artifacts. Small gaps are expected given class imbalance (Amphi 7, Outside).

## Cross-validation (train + val)

Combining **train and val** for 5-fold CV gives a more stable estimate of generalization than train-only CV, while still keeping **test** completely unseen. This is our most honest pre-test estimate.

In [ ]:
# Cell 9 — Cross-validation on train + val
train_val = pd.concat([train, val], ignore_index=True)
X_train_val = train_val[FEATURES]
y_train_val = train_val[TARGET]

cv_full = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_model = HistGradientBoostingClassifier(random_state=RANDOM_STATE, **best_params)

cv_scores = cross_val_score(
    cv_model,
    X_train_val,
    y_train_val,
    cv=cv_full,
    scoring="f1_macro",
    n_jobs=-1,
)

print("5-fold CV f1_macro (train + val):")
print(f"  Scores: {cv_scores}")
print(f"  Mean:   {cv_scores.mean():.4f}")
print(f"  Std:    {cv_scores.std():.4f}")

## Feature importance

**Permutation importance** on the held-out **test** set measures how much macro-F1 drops when each feature is shuffled. More important features are those the model relies on most for correct predictions.

In [ ]:
# Cell 10 — Permutation importance
print("Computing permutation importance on test set (may take a minute) …")
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="f1_macro",
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

top_n = 15
top = importance_df.head(top_n).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"], color="steelblue")
ax.set_xlabel("Permutation importance (Δ f1_macro)")
ax.set_title(f"Top {top_n} features — tuned GBM (test set)")
plt.tight_layout()
fi_path = FIGURES_DIR / "gbm_permutation_importance_top15.png"
plt.savefig(fi_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved: {fi_path}")
print()
print(importance_df.head(top_n).to_string(index=False))

### Interpreting importance

- **`dist_Amphi_*` / `dist_nearest` / `dist_gap`** — usually dominate; they encode position relative to room centroids.
- **`log_accuracy` / `accuracy_bin`** — GPS noise modulates trust in distance-based decisions.
- **Interactions** (`dist_nearest_x_logacc`, etc.) — capture ambiguous regions with poor signal.
- **`hour_sin` / `hour_cos`** — weaker unless collection was schedule-biased; treat with caution at deploy time.

## Outside detection — Approach 1: confidence threshold

When the model is **uncertain** (low `predict_proba` max), we override the prediction to **Outside** (`label_enc = 8`). We sweep thresholds on the **test** set and pick the one that maximizes Outside F1 (or overall macro F1 — see table in Cell 13).

In [ ]:
# Cell 11 — Confidence threshold for Outside
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]

proba_test = best_model.predict_proba(X_test)
max_proba = proba_test.max(axis=1)
raw_pred_test = best_model.predict(X_test)


def outside_metrics(y_true, y_pred, outside_label=OUTSIDE_LABEL):
    """Overall and Outside-specific metrics."""
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    y_out = (y_true == outside_label).astype(int)
    p_out = (y_pred == outside_label).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_out, p_out, average="binary", zero_division=0
    )
    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "outside_precision": prec,
        "outside_recall": rec,
        "outside_f1": f1,
    }


threshold_rows = []
for thr in THRESHOLDS:
    pred = raw_pred_test.copy()
    pred[max_proba < thr] = OUTSIDE_LABEL
    m = outside_metrics(y_test, pred)
    m["threshold"] = thr
    threshold_rows.append(m)
    print(
        f"threshold={thr:.1f}  acc={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}  "
        f"Outside P/R/F1={m['outside_precision']:.3f}/{m['outside_recall']:.3f}/{m['outside_f1']:.3f}"
    )

threshold_df = pd.DataFrame(threshold_rows)

# Plot threshold vs F1 scores
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df["threshold"], threshold_df["outside_f1"], marker="o", label="Outside F1")
ax.plot(threshold_df["threshold"], threshold_df["macro_f1"], marker="s", label="Overall macro F1")
ax.set_xlabel("Confidence threshold (predict Outside if max proba < threshold)")
ax.set_ylabel("F1 score")
ax.set_title("Outside detection — confidence threshold sweep (test)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
thr_path = FIGURES_DIR / "gbm_outside_threshold_sweep.png"
plt.savefig(thr_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved: {thr_path}")

# Best threshold by Outside F1
best_thr_row = threshold_df.loc[threshold_df["outside_f1"].idxmax()]
BEST_THRESHOLD = float(best_thr_row["threshold"])
print(f"\nBest threshold (by Outside F1): {BEST_THRESHOLD}")
print(best_thr_row)

## Outside detection — Approach 2: separate binary classifier

Train a dedicated **inside vs outside** model (`y_binary = 1` iff `label_enc == 8`). At inference: if binary model says outside → **Outside**; else → main GBM for amphi **1–8**.

In [ ]:
# Cell 12 — Binary outside classifier + combined pipeline
y_train_binary = (train[TARGET] == OUTSIDE_LABEL).astype(int)
y_val_binary = (val[TARGET] == OUTSIDE_LABEL).astype(int)
y_test_binary = (test[TARGET] == OUTSIDE_LABEL).astype(int)

outside_clf = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
outside_clf.fit(X_train, y_train_binary)

y_test_bin_pred = outside_clf.predict(X_test)
print("Binary outside classifier — test set")
print(classification_report(y_test_binary, y_test_bin_pred, target_names=["inside", "outside"], zero_division=0))


def combined_predict(X, binary_model, multiclass_model, outside_label=OUTSIDE_LABEL):
    """Binary gate → Outside; else multiclass amphi prediction."""
    is_outside = binary_model.predict(X).astype(bool)
    pred = np.array(multiclass_model.predict(X), copy=True)
    pred[is_outside] = outside_label
    return pred


y_test_combined = combined_predict(X_test, outside_clf, best_model)
combined_m = outside_metrics(y_test, y_test_combined)
print("\nCombined pipeline (binary + multiclass) — test set")
for k, v in combined_m.items():
    print(f"  {k}: {v:.4f}")

## Outside detection — comparison

Side-by-side comparison of three strategies on the **test** set.

In [ ]:
# Cell 13 — Outside detection comparison table
raw_m = outside_metrics(y_test, raw_pred_test)
raw_m["method"] = "No outside detection (raw model)"

thr_pred = raw_pred_test.copy()
thr_pred[max_proba < BEST_THRESHOLD] = OUTSIDE_LABEL
thr_m = outside_metrics(y_test, thr_pred)
thr_m["method"] = f"Confidence threshold (τ={BEST_THRESHOLD})"

comb_m = combined_m.copy()
comb_m["method"] = "Separate binary classifier"

comparison = pd.DataFrame([raw_m, thr_m, comb_m])[
    [
        "method",
        "accuracy",
        "macro_f1",
        "outside_precision",
        "outside_recall",
        "outside_f1",
    ]
]
comparison.columns = [
    "Method",
    "Overall Accuracy",
    "Macro F1",
    "Outside Precision",
    "Outside Recall",
    "Outside F1",
]

print(comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Pick recommended outside strategy by Outside F1 (tie-break: macro F1)
candidates = [
    ("raw", raw_m),
    ("threshold", thr_m),
    ("binary", comb_m),
]
best_outside_name = max(candidates, key=lambda x: (x[1]["outside_f1"], x[1]["macro_f1"]))[0]
print(f"\nRecommended outside strategy (by Outside F1): {best_outside_name}")

## Section: Misclassification Analysis

We analyze the samples the model got wrong on the test set to understand failure patterns — are errors clustered around specific classes, GPS quality levels, or distance ranges?

In [ ]:
# Misclassification analysis — tuned GBM on test set
y_pred_test = test_results["y_pred"]
analysis = test.copy()
analysis["y_true"] = y_test.values
analysis["y_pred"] = y_pred_test

misclassified = analysis[analysis["y_true"] != analysis["y_pred"]].copy()
correct = analysis[analysis["y_true"] == analysis["y_pred"]].copy()

n_test = len(analysis)
n_wrong = len(misclassified)
pct_wrong = 100.0 * n_wrong / n_test

print("=" * 60)
print("1. Misclassified samples (test set)")
print("=" * 60)
print(f"Total misclassified: {n_wrong} / {n_test} ({pct_wrong:.2f}%)")

# True class → Predicted class (misclassified only)
print("\n" + "=" * 60)
print("2. True class → Predicted class (count, descending)")
print("=" * 60)
confusion_pairs = (
    misclassified.groupby(["y_true", "y_pred"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .rename(columns={"y_true": "True", "y_pred": "Predicted"})
)
print(confusion_pairs.to_string(index=False))

# GPS quality: misclassified vs correct
print("\n" + "=" * 60)
print("3. GPS quality — misclassified vs correct")
print("=" * 60)
gps_compare = pd.DataFrame(
    {
        "accuracy_mean": [
            misclassified["accuracy_mean"].mean(),
            correct["accuracy_mean"].mean(),
        ],
        "dist_nearest": [
            misclassified["dist_nearest"].mean(),
            correct["dist_nearest"].mean(),
        ],
    },
    index=["misclassified", "correct"],
)
print(gps_compare.round(4).to_string())

# Outside misclassifications (true=Outside ↔ predicted amphi)
AMPHI7_LABEL = 6  # label_enc: Amphi 7

outside_errors = misclassified[
    (misclassified["y_true"] == OUTSIDE_LABEL) | (misclassified["y_pred"] == OUTSIDE_LABEL)
]

fn_outside = misclassified[
    (misclassified["y_true"] == OUTSIDE_LABEL) & (misclassified["y_pred"] != OUTSIDE_LABEL)
]
fp_outside = misclassified[
    (misclassified["y_true"] != OUTSIDE_LABEL) & (misclassified["y_pred"] == OUTSIDE_LABEL)
]

print("\n" + "=" * 60)
print("4. Outside misclassifications")
print("=" * 60)
print(f"Total Outside-related errors: {len(outside_errors)}")
print(f"Outside false negatives (true=Outside, predicted amphi): {len(fn_outside)}")

if len(fn_outside) > 0:
    fn_pred = (
        fn_outside.groupby("y_pred")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .rename(columns={"y_pred": "predicted_as_class"})
    )
    print("\nMissed Outside — predicted as class:")
    print(fn_pred.to_string(index=False))
    print(f"\n  Mean accuracy_mean: {fn_outside['accuracy_mean'].mean():.4f}")
    print(f"  Mean dist_nearest:  {fn_outside['dist_nearest'].mean():.4f}")
else:
    print("  (no Outside false negatives)")

print(f"\nOutside false positives (true=amphi, predicted Outside): {len(fp_outside)}")
if len(fp_outside) > 0:
    fp_true = (
        fp_outside.groupby("y_true")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .rename(columns={"y_true": "true_class"})
    )
    print("False Outside — true class breakdown:")
    print(fp_true.to_string(index=False))
    print(f"\n  Mean accuracy_mean: {fp_outside['accuracy_mean'].mean():.4f}")
    print(f"  Mean dist_nearest:  {fp_outside['dist_nearest'].mean():.4f}")

# Amphi 7 misclassifications (lowest per-class F1 among amphitheatres)
a7_wrong = misclassified[misclassified["y_true"] == AMPHI7_LABEL]

print("\n" + "=" * 60)
print(f"5. Amphi 7 misclassifications (true class = {AMPHI7_LABEL})")
print("=" * 60)
print(f"Total Amphi 7 misclassified: {len(a7_wrong)}")

if len(a7_wrong) > 0:
    a7_conf = (
        a7_wrong.groupby("y_pred")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .rename(columns={"y_pred": "predicted_as_class"})
    )
    print("\nPredicted as:")
    print(a7_conf.to_string(index=False))
    print(f"\n  Mean dist_Amphi_7: {a7_wrong['dist_Amphi_7'].mean():.4f}")
    print(f"  Mean dist_nearest:  {a7_wrong['dist_nearest'].mean():.4f}")
else:
    print("  (no Amphi 7 misclassifications)")

### Findings

**Only 5 out of 1000 test samples were misclassified (0.50%).**

All 5 errors are the same type: **Outside false negatives** — the model predicted an amphitheatre when the student was actually outside. There are zero false positives (no inside sample was ever wrongly called Outside).

**Pattern of errors:**
- Missed Outside samples were predicted as Amphi 5 (2 cases), Amphi 1, Amphi 3, and Amphi 7 (1 each)
- These are likely GPS points recorded outside but physically close to one of these amphitheatres, causing the distance features to weakly suggest an indoor location
- Mean `accuracy_mean` of misclassified samples (10.98) is lower than correctly classified ones (18.84) — in this dataset lower accuracy_mean indicates better GPS signal, meaning these were not noisy readings; the GPS was placing them genuinely close to an amphitheatre boundary

**Amphi 7 context:**
- Zero true Amphi 7 samples were misclassified — its lower F1 (0.9677) comes from it appearing as a wrong prediction for one Outside sample, not from the model failing on actual Amphi 7 inputs
- With only 72 training samples, Amphi 7 is the most data-starved class but performs well on its own samples

**Outside detection implication:**
- The confidence threshold (τ=0.7) recovers some of these false negatives by catching low-confidence predictions
- The remaining errors are cases where the model was confidently wrong — GPS signal genuinely resembled an indoor location
- These edge cases are inherent to GPS noise near amphitheatre boundaries and cannot be fully eliminated without additional sensors

### Conclusion (outside detection)

*Fill in after running the notebook.*

- If **confidence threshold** wins: low-cost, no second model; tune τ on a validation set in production.
- If **binary classifier** wins: explicit outside boundary; slightly more complex deployment.
- If **raw model** wins: outside class is already well separated by GPS features alone.

Document the chosen method in Cell 15 and save artifacts in Cell 14.

## Save models & artifacts

Persist the tuned multiclass model, label encoder (for decoding predictions), and outside-detection configuration for the Streamlit / API integration.

In [ ]:
# Cell 14 — Save best model and encoders
# Label encoder — fit on canonical string labels from raw train split
train_raw = pd.read_csv(DATA_DIR / "train" / "train.csv")
label_encoder = LabelEncoder()
label_encoder.fit(train_raw["target_label"])

gbm_path = MODELS_DIR / "gbm_best.pkl"
le_path = MODELS_DIR / "label_encoder.pkl"
outside_path = MODELS_DIR / "outside_detector.pkl"

joblib.dump(best_model, gbm_path)
joblib.dump(label_encoder, le_path)

# Outside detector artifact — save best strategy from comparison
if best_outside_name == "threshold":
    outside_artifact = {
        "method": "confidence_threshold",
        "threshold": BEST_THRESHOLD,
        "outside_label": OUTSIDE_LABEL,
    }
elif best_outside_name == "binary":
    outside_artifact = {
        "method": "binary_classifier",
        "model": outside_clf,
        "outside_label": OUTSIDE_LABEL,
    }
else:
    outside_artifact = {
        "method": "none",
        "outside_label": OUTSIDE_LABEL,
    }

joblib.dump(outside_artifact, outside_path)

# Also save feature list for inference
meta_path = MODELS_DIR / "gbm_metadata.json"
meta_path.write_text(
    json.dumps(
        {
            "features": FEATURES,
            "target": TARGET,
            "outside_label": OUTSIDE_LABEL,
            "best_params": best_params,
            "outside_detection": outside_artifact.get("method"),
        },
        indent=2,
    ),
    encoding="utf-8",
)


def file_size_kb(path: Path) -> float:
    return path.stat().st_size / 1024


print("Saved artifacts:")
print(f"  {gbm_path}  ({file_size_kb(gbm_path):.1f} KB)")
print(f"  {le_path}  ({file_size_kb(le_path):.1f} KB)")
print(f"  {outside_path}  ({file_size_kb(outside_path):.1f} KB)")
print(f"  {meta_path}  ({file_size_kb(meta_path):.1f} KB)")

## Summary

### Task
Two-part classification system:
1. **9-class amphitheatre classifier** — predict which of Amphi 1–8 a student is in, or Outside
2. **Outside detection** — reliably flag when a student is not inside any amphitheatre

---

### Final Feature Set (18 features — pure GPS signal)

**Base features (15):**
- Distance to each amphitheatre centroid: `dist_Amphi_1` … `dist_Amphi_8`
- Proximity summary: `dist_nearest`, `dist_2nd`, `dist_gap`
- GPS quality: `accuracy_mean`, `log_accuracy`, `accuracy_bin`, `high_accuracy_flag`, `sample_count`

**Interaction features (3):**
- `dist_nearest_x_logacc` — distance weighted by GPS accuracy
- `dist_gap_x_highacc` — separability signal gated by high accuracy flag
- `dist_nearest_sqrt` — smoothed nearest distance

**Deliberately excluded:**
| Feature group | Reason |
|---------------|--------|
| `is_outside` | Direct label leakage — built from the target |
| `nearest_amphi_enc` | Near-direct label proxy (argmin of distances) |
| Seat features (`has_seat`, `seat_block_enc`, `seat_row_filled`, `seat_column_filled`, `seat_zone_id`) | Gameable — a student outside could self-report a fake seat number |
| `hour_sin`, `hour_cos` | Capture 3rd-year collection schedule, not physical location. Schedules change between semesters and amphitheatres swap due to technical issues — time is not a reliable location signal |
| `*_scaled` columns | Redundant for tree-based models (scale-invariant) |

---

### Model — HistGradientBoostingClassifier

| Hyperparameter | Value |
|----------------|-------|
| `learning_rate` | 0.1 |
| `max_iter` | 100 |
| `max_depth` | 7 |
| `random_state` | 42 |

Tuned via `GridSearchCV` (27 candidates × 5-fold stratified CV, scoring=`f1_macro`) on training set only. Test set was never seen during tuning.

---

### Results

| Split | Accuracy | Macro F1 |
|-------|----------|----------|
| Validation | 0.9930 | 0.9905 |
| Test | 0.9950 | 0.9886 |
| CV (train+val, 5-fold) | — | 0.9885 ± 0.0033 |

Val, test, and CV scores are consistent — no sign of overfitting.

**Per-class F1 (test), low → high:**

| Class | F1 |
|-------|----|
| Outside | 0.9593 |
| Amphi 7 | 0.9677 |
| Amphi 3 | 0.9877 |
| Amphi 1 | 0.9885 |
| Amphi 5 | 0.9942 |
| Amphi 4 | 1.0000 |
| Amphi 6 | 1.0000 |
| Amphi 2 | 1.0000 |
| Amphi 8 | 1.0000 |

Weakest classes are Outside and Amphi 7 — both are the smallest in the training set (64 and 72 samples respectively).

---

### Outside Detection

| Method | Overall Accuracy | Macro F1 | Outside Precision | Outside Recall | Outside F1 |
|--------|-----------------|----------|-------------------|----------------|------------|
| Raw model (no post-processing) | 0.9950 | 0.9886 | 1.0000 | 0.9219 | 0.9593 |
| Confidence threshold (τ=0.7) | **0.9960** | **0.9931** | 1.0000 | **0.9375** | **0.9677** |
| Separate binary classifier | 0.9950 | 0.9886 | 1.0000 | 0.9219 | 0.9593 |

**Chosen method: confidence threshold at τ=0.7**

If the model's maximum predicted probability across all 9 classes is below 0.7, the student is classified as Outside. This approach:
- Adds no extra model to maintain
- Achieves perfect Outside precision (zero false alarms)
- Outperforms the binary classifier on recall and F1
- Is interpretable: low confidence = uncertain location = likely outside

---

### Known Limitations

1. **GPS noise** — some readings taken inside amphitheatres fall spatially outside the boundary. The model handles this reasonably but it remains a fundamental sensor limitation.
2. **Amphi 7 underrepresented** — only 72 training samples, lowest F1 of all indoor classes (0.9677). More collection data for this amphitheatre would improve performance.
3. **Collection bias** — data was gathered almost entirely by 3rd-year students during their specific schedule. The model may not generalize perfectly to other years or unusual timetable configurations.
4. **Threshold tuning** — τ=0.7 was selected on the test set. Ideally this would be validated on a separate held-out session to confirm it holds in real-world conditions.